# The EYE/N3 certificate checker, in a notebook

This drives `eye/check.py` — the N3 re-implementation of open-ontologies'
`oo-cert` — from Python, on certificates the engine would write.

What it shows, in order:

1. an honest certificate is **accepted** (exit 0);
2. a forged one is **refused** (exit 1), naming the first bad step;
3. the two certificate *kinds* (`oo-cert` OWL-RL vs `oo-horn`);
4. the optional **proven anchor** (Rocq), and a real false pass it catches.

**What an `accept` means here.** The EYE checker carries **no machine-checked
theorem** (`OOCert.certificate_sound` is about the Lean function and is not
reproved here). An accept means *the rules licensed every step*, not
*it is entailed*. With the Rocq anchor set — and only for `oo-horn` certificates —
the accept gains the theorem. Section 4 makes that difference concrete.

## 0 · Locate the checker

In [1]:
import os, subprocess, json, sys, shutil
from pathlib import Path

# the notebook lives in eye/examples/; the checker is one level up
EYE = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
CHECK = EYE / "check.py"
O = EYE.parent                      # the open-ontologies checkout
FIX = O / "tests" / "fixtures"
ROCQ = O / "rocq" / "build" / "oo-horn-rocq"
assert CHECK.exists(), f"no check.py at {CHECK}"
npx = shutil.which("npx")
print("checker :", CHECK)
print("fixtures:", FIX)
print("npx     :", npx or "MISSING (the EYE checker runs via eyereasoner)")
print("anchor  :", ROCQ, "(built)" if ROCQ.exists() else "(not built)")

checker : /a0/usr/projects/pup/open-ontologies/eye/check.py
fixtures: /a0/usr/projects/pup/open-ontologies/tests/fixtures
npx     : /usr/bin/npx
anchor  : /a0/usr/projects/pup/open-ontologies/rocq/build/oo-horn-rocq (built)


## 1 · Run the checker

`check.py ASSERTED.tsv DERIVATIONS.tsv` — the same CLI shape and exit codes as
`lake exe oo-cert`: `0` accepted, `1` a step rejected (JSON names it),
`2` unreadable/unparseable. The helper never decides anything; it runs the
checker and returns exactly what it printed. Exit `3` appears only with the
anchor set, and only when the two disagree (section 4).

In [2]:
def run_checker(asserted, derivations, env=None):
    """Run eye/check.py and return (exit_code, parsed_json_or_raw)."""
    e = dict(os.environ)
    if env: e.update(env)
    p = subprocess.run([sys.executable, str(CHECK), str(asserted), str(derivations)],
                       capture_output=True, text=True, cwd=str(EYE), env=e)
    line = next((l for l in p.stdout.splitlines() if l.startswith("{")), "")
    try: out = json.loads(line) if line else {"raw": p.stdout[-300:], "stderr": p.stderr[-300:]}
    except json.JSONDecodeError: out = {"raw": line}
    return p.returncode, out

def show(label, asserted, derivations, env=None):
    code, out = run_checker(asserted, derivations, env)
    print(f"{label:<44} exit={code}")
    print("   ", json.dumps(out))
    return code, out

### 1a · An honest certificate is accepted

In [3]:
show("honest rdfs9 certificate",
     FIX / "refute/asserted.tsv",
     FIX / "refute/derivations.tsv")

honest rdfs9 certificate                     exit=0
    {"ok": true, "asserted": 4, "derivations": 1}


(0, {'ok': True, 'asserted': 4, 'derivations': 1})

### 1b · A forged one is refused

Take the same certificate and change **only the conclusion object**
(`Carnivore` -> `Herbivore`), leaving both premises exactly as they were. The
checker refuses it and names the first bad step. This is the page's red edge:
same premises, one changed line, caught.

In [4]:
import re
src = (FIX / "refute/derivations.tsv").read_text()
forged = re.sub("Carnivore", "Herbivore", src, count=1)   # first occurrence only
forged_path = EYE / "examples" / "forged_derivations.tsv"
forged_path.write_text(forged)
print("before:", src.splitlines()[0].split("\t")[:4])
print("after :", forged.splitlines()[0].split("\t")[:4])
print()
show("forged certificate", FIX / "refute/asserted.tsv", forged_path)

before: ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Carnivore>']
after : ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Herbivore>']



forged certificate                           exit=1
    {"ok": false, "rejected_step": 0, "rule": "rdfs9", "conclusion": "<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>", "premises": ["<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Lion>", "<http://ex.org/Lion> <http://www.w3.org/2000/01/rdf-schema#subClassOf> <http://ex.org/Carnivore>"]}


(1,
 {'ok': False,
  'rejected_step': 0,
  'rule': 'rdfs9',
  'conclusion': '<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>',
  'premises': ['<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Lion>',
   '<http://ex.org/Lion> <http://www.w3.org/2000/01/rdf-schema#subClassOf> <http://ex.org/Carnivore>']})

## 2 · Two certificate kinds

The checker auto-detects the format:

- **`oo-cert`** — `derivations.tsv`: `rule TAB s TAB p TAB o`, tab, premises. The
  OWL-RL rule set (29 arms) — what the engine's `reason --certificate` writes and
  what the *Try It* page checks.
- **`oo-horn`** — `horn.tsv`: `idx TAB bindCount TAB bindings TAB conclusion TAB
  premises`, the Horn rule set (the Rocq-proven one).

In [5]:
show("oo-horn (Horn rule set)",
     FIX / "horn/asserted.tsv",
     FIX / "horn/good.tsv")

oo-horn (Horn rule set)                      exit=0
    {"ok": true, "asserted": 2, "derivations": 1}


(0, {'ok': True, 'asserted': 2, 'derivations': 1})

## 3 · Shape and order are enforced — run the suite

The checker rejects: a premise asserted nowhere and concluded nowhere; premises
in the **wrong order**; `scm-avf2` with the conclusion in the *natural*
direction (the one rule whose conclusion is deliberately reversed — the natural
way round would be unsound); `cls-int1` when one list member's type is missing.

The suite that proves all of this is `eye/run_tests.sh`:

In [6]:
p = subprocess.run(["bash", str(EYE / "run_tests.sh")], capture_output=True, text=True, cwd=str(EYE))
print("\n".join(p.stdout.splitlines()[-8:]))
print("suite exit:", p.returncode)

       {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations": 1, "theorem": "OOCertRocq.entails_of_builtin_horn", "means": "every conclusion is true in every model of the asserted graph"}}}
PASS [anchor: oo-horn bad_conclusion (caught false pass)] exit=3
       {"ok": false, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 1, "verdict": {"ok": false, "kernel": "rocq", "rules": 27, "asserted": 2, "derivations": 1}}, "anchor_disagrees": true, "error": "the proven oo-horn anchor refused a certificate EYE accepted (possible false pass)"}
PASS [anchor: oo-cert declines, EYE verdict stands] exit=0
       {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}
PASS [anchor: broken p

## 4 · The optional proven anchor (Rocq), and a false pass it caught

Setting `OO_HORN_ROCQ` + `OO_RULES` runs the **Rocq-extracted, machine-checked**
`oo-horn` checker (theorem `entails_of_builtin_horn`) over the same inputs:

- when it **agrees**, the accept carries the theorem name;
- when it **disagrees** with EYE on an `oo-horn` certificate, the exit code is
  **3**, not 0 — a caught false pass is not a clean accept.

It speaks for `oo-horn` only; on an `oo-cert` certificate it **declines** rather
than overclaim.

In [7]:
if ROCQ.exists():
    ENV = {"OO_HORN_ROCQ": str(ROCQ), "OO_RULES": str(FIX / "horn/builtin_rules.tsv")}
    print("— oo-horn good: EYE and the anchor agree —")
    show("oo-horn good + anchor", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv", ENV)
    print()
    print("— oo-horn bad_conclusion: EYE accepts, the theorem refuses (exit 3) —")
    show("oo-horn bad_conclusion + anchor", FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv", ENV)
    print()
    print("— oo-cert: the anchor declines (Horn-only); the EYE verdict stands —")
    show("oo-cert + anchor (declines)", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv", ENV)
else:
    print("The Rocq anchor is not built here. Build it with:")
    print("  (cd", O / "rocq", "&& bash build.sh)")

— oo-horn good: EYE and the anchor agree —


oo-horn good + anchor                        exit=0
    {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations": 1, "theorem": "OOCertRocq.entails_of_builtin_horn", "means": "every conclusion is true in every model of the asserted graph"}}}

— oo-horn bad_conclusion: EYE accepts, the theorem refuses (exit 3) —


oo-horn bad_conclusion + anchor              exit=3
    {"ok": false, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 1, "verdict": {"ok": false, "kernel": "rocq", "rules": 27, "asserted": 2, "derivations": 1}}, "anchor_disagrees": true, "error": "the proven oo-horn anchor refused a certificate EYE accepted (possible false pass)"}

— oo-cert: the anchor declines (Horn-only); the EYE verdict stands —


oo-cert + anchor (declines)                  exit=0
    {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}


### Why `bad_conclusion` matters

The fixture carries a binding `b = Z` that contradicts its own conclusion object
`B`. EYE **accepts** it (the Horn bindings are stripped when the rule index is
mapped through `asHorn`); the proven checker **refuses** it. That is a textbook
**false pass** — the dangerous direction — caught by running the two together.
Without the anchor, `check.py` returns exit 0; with it, exit 3 and
`"anchor_disagrees": true`. The test suite keeps this under test.

## 5 · What this does *not* claim

- The `oo-cert` soundness theorem (`OOCert.certificate_sound`) is about the
  **Lean** function and is **not** reproved here. For `oo-cert`, an accept is
  *the rules licensed every step*.
- The Rocq anchor proves a theorem about the **Horn** rule set only
  (`entails_of_builtin_horn`), so it applies to `oo-horn`, not `oo-cert`.
- Even with the anchor, the verdict is **conditional on the asserted graph**:
  neither EYE nor Rocq checks the graph is the one the engine actually reasoned
  over.

That is the honest boundary of this notebook, and of `eye/` itself.